# Shared recommender experiment — wiring check

**SYNTHETIC ONLY.** This notebook verifies the two-package contract, not forgetting in a research model. The counting model is an independent-domain isolation control: learning B cannot change A scores.

Read `docs/PACKAGES-AND-BUILDING.md` and `docs/MEETING-2026-10-15.md` before extending it. The research model, data/filter, task definition, cutoff and budget still need human decisions. The choices below are explicit demonstration settings. Restart the kernel and Run All; no cell contains a private split, metric or training loop.

In [ ]:
from pathlib import Path
from uuid import uuid4

from cp4285.experiment import SyntheticSource, plot_results, run
from cp4285.models import CountingRecommender

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
output = root / 'runs' / f'synthetic-contract-{uuid4().hex[:12]}'
initial_epochs = 3
k = 1  # Tiny two-item fixture; not the final study's metric cutoff.

## Inspect the shared inputs
The loader owns frozen queries/targets/candidates and independent A-only, mixed, B-only and changing-mixture schedules. Each update block contains four supervised targets. `B share` means B / (A+B), not B/A. Histories are never spliced across unrelated users.

In [ ]:
data = SyntheticSource().load_data()
data.validate()
data

In [ ]:
model = CountingRecommender()
result = run(model, data, output=output, initial_epochs=initial_epochs, k=k)
result['manifest'], result['selected_epoch']

## Inspect evidence, then plot saved results
The runner selects the first best validation-NDCG epoch, forks each condition from that state, and evaluates exactly the same tests after every block. `targets_seen` exposes realized domain counts. `prepared.json`, `manifest.json` and `results.json` are create-only artifacts in the printed directory. No durable model checkpoint/resume is implemented yet.

The graphs read `results.json`, not a notebook-local reinterpretation of the scores. For a single target, Recall@k = Hit@k. Improvement, stability and decline are all valid outcomes; these invented curves are not evidence about Amazon or neural forgetting.

In [ ]:
print(output)
result['records']

In [ ]:
plot_results(output / 'results.json', task='retention-A')

In [ ]:
plot_results(output / 'results.json', task='adaptation-B')

## Fan out, then converge
Implement a `DataSource` or `Recommender`, with a short Markdown account of the question, evidence, decisions and limitations. Import it here; do not change the benchmark to suit a model. Before comparison, agree one prepared snapshot, candidate policy, initial-selection rule, exposure budget and metric cutoff. Capacity studies must state what changes and report parameter counts. See the copyable team prompt in `docs/PACKAGES-AND-BUILDING.md`.